## 1. Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

In [2]:
QUARTER = "2025Q1"

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / QUARTER
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data     : {RAW_DATA_DIR}")
print(f"Processed    : {PROCESSED_DATA_DIR}")

Project root : C:\Users\Krish\Desktop\BU\Codes\Projects\ADR\adr-prediction-system
Raw data     : C:\Users\Krish\Desktop\BU\Codes\Projects\ADR\adr-prediction-system\data\raw\2025Q1
Processed    : C:\Users\Krish\Desktop\BU\Codes\Projects\ADR\adr-prediction-system\data\processed


In [3]:
demo_path = RAW_DATA_DIR / f"demo{QUARTER.lower()}.csv"
drug_path = RAW_DATA_DIR / f"drug{QUARTER.lower()}.csv"
reac_path = RAW_DATA_DIR / f"reac{QUARTER.lower()}.csv"

demo = pd.read_csv(demo_path, low_memory=False)
drug = pd.read_csv(drug_path, low_memory=False)
reac = pd.read_csv(reac_path, low_memory=False)

print("FAERS data loaded successfully.\n")
print(f"DEMO : {demo.shape}")
print(f"DRUG : {drug.shape}")
print(f"REAC : {reac.shape}")

FAERS data loaded successfully.

DEMO : (400514, 33)
DRUG : (2008162, 20)
REAC : (1432926, 4)


In [4]:
print("DEMO columns:")
print(demo.columns.tolist(), "\n")

print("DRUG columns:")
print(drug.columns.tolist(), "\n")

print("REAC columns:")
print(reac.columns.tolist(), "\n")

DEMO columns:
['primaryid', 'caseid', 'caseversion', 'i_f_code', 'i_f_code_num', 'event_dt', 'event_dt_num', 'mfr_dt', 'mfr_dt_num', 'init_fda_dt', 'init_fda_dt_num', 'fda_dt', 'fda_dt_num', 'rept_cod', 'rept_cod_num', 'auth_num', 'mfr_num', 'mfr_sndr', 'lit_ref', 'age', 'age_cod', 'age_grp', 'sex', 'e_sub', 'wt', 'wt_cod', 'rept_dt', 'rept_dt_num', 'to_mfr', 'occp_cod', 'reporter_country', 'occr_country', 'occp_cod_num'] 

DRUG columns:
['primaryid', 'caseid', 'drug_seq', 'role_cod', 'drugname', 'prod_ai', 'val_vbm', 'route', 'dose_vbm', 'cum_dose_chr', 'cum_dose_unit', 'dechal', 'rechal', 'lot_num', 'exp_dt', 'nda_num', 'dose_amt', 'dose_unit', 'dose_form', 'dose_freq'] 

REAC columns:
['primaryid', 'caseid', 'pt', 'drug_rec_act'] 



In [5]:
def missing_summary(df, columns):
    """Return a table with the count and percentage of missing values for each column."""
    summary = pd.DataFrame({
        "column": columns,
        "missing_count": [df[col].isna().sum() for col in columns],
    })
    summary["missing_percent"] = (summary["missing_count"] / len(df) * 100).round(2)
    return summary

demo_candidates = [
    "age", "age_cod", "age_grp", "sex", "wt", "wt_cod",
    "reporter_country", "occr_country", "occp_cod",
]

print("DEMO CANDIDATE FEATURES")
display(missing_summary(demo, demo_candidates))

drug_candidates = [
    "drugname", "role_cod", "route", "dose_vbm", "cum_dose_chr", "cum_dose_unit",
    "dose_amt", "dose_unit", "dose_form", "dose_freq", "prod_ai", "val_vbm",
    "dechal", "rechal",
]

print("\nDRUG CANDIDATE FEATURES")
display(missing_summary(drug, drug_candidates))

DEMO CANDIDATE FEATURES


,column,missing_count,missing_percent
0,age,156970,39.19
1,age_cod,156950,39.19
2,age_grp,259903,64.89
3,sex,81778,20.42
4,wt,331751,82.83
5,wt_cod,331752,82.83
6,reporter_country,1,0.00
7,occr_country,16127,4.03
8,occp_cod,32953,8.23



DRUG CANDIDATE FEATURES


,column,missing_count,missing_percent
0,drugname,0,0.00
1,role_cod,0,0.00
2,route,651204,32.43
3,dose_vbm,556861,27.73
4,cum_dose_chr,1974204,98.31
5,cum_dose_unit,1974204,98.31
6,dose_amt,1198219,59.67
7,dose_unit,1198219,59.67
8,dose_form,1019864,50.79
9,dose_freq,1588500,79.10


In [6]:
print("ROLE COD VALUES")
print(drug["role_cod"].value_counts(dropna=False))

print("\nPROD_AI EXAMPLES")
print(drug["prod_ai"].value_counts(dropna=False).head(20))

print("\nVAL_VBM VALUES")
print(drug["val_vbm"].value_counts(dropna=False))

print("\nDOSE FORM VALUES")
print(drug["dose_form"].value_counts(dropna=False).head(20))

ROLE COD VALUES
role_cod
SS    844949
C     720718
PS    431702
I      10779
DN        14
Name: count, dtype: int64

PROD_AI EXAMPLES
prod_ai
TIRZEPATIDE                 52320
DUPILUMAB                   47770
INFLIXIMAB-DYYB             35741
NaN                         31933
PREDNISONE                  28740
METHOTREXATE                23840
ACETAMINOPHEN               23654
TOCILIZUMAB                 21698
VEDOLIZUMAB                 21296
RITUXIMAB                   20019
ADALIMUMAB                  16512
OMALIZUMAB                  16430
INFLIXIMAB                  16107
CETIRIZINE HYDROCHLORIDE    15837
SULFASALAZINE               14074
FOLIC ACID                  13800
ALBUTEROL SULFATE           12641
ASPIRIN                     12072
TOFACITINIB CITRATE         12069
LEUPROLIDE ACETATE          11867
Name: count, dtype: int64

VAL_VBM VALUES
val_vbm
1    1982317
2      25845
Name: count, dtype: int64

DOSE FORM VALUES
dose_form
NaN                                             

In [7]:
rows_before = len(demo)

demo = (
    demo.sort_values(["caseid", "caseversion", "fda_dt"])
    .drop_duplicates(subset="caseid", keep="last")
    .reset_index(drop=True)
)

kept_ids = set(demo["primaryid"])
drug = drug[drug["primaryid"].isin(kept_ids)].copy()
reac = reac[reac["primaryid"].isin(kept_ids)].copy()

print(f"Reports before de-duplication : {rows_before:,}")
print(f"Reports after de-duplication  : {len(demo):,}")
print(f"Old case versions removed     : {rows_before - len(demo):,}")
print(f"Unique caseids now            : {demo['caseid'].nunique():,}")

Reports before de-duplication : 400,514
Reports after de-duplication  : 400,514
Old case versions removed     : 0
Unique caseids now            : 400,514


In [8]:
demo_features = ["age", "age_cod", "sex"]

drug_features = ["drugname", "prod_ai", "role_cod", "route"]

target_columns = [
    "Diarrhoea", "Nausea", "Vomiting", "Fatigue", "Headache",
    "Dizziness", "Rash", "Dyspnoea", "Abdominal pain", "Constipation",
]

print("FINAL FEATURE SELECTION")
print("\nDEMO features:")
for feature in demo_features:
    print(f"  - {feature}")

print("\nDRUG features:")
for feature in drug_features:
    print(f"  - {feature}")

print("\nTARGETS:")
for target in target_columns:
    print(f"  -> {target}")

FINAL FEATURE SELECTION

DEMO features:
  - age
  - age_cod
  - sex

DRUG features:
  - drugname
  - prod_ai
  - role_cod
  - route

TARGETS:
  -> Diarrhoea
  -> Nausea
  -> Vomiting
  -> Fatigue
  -> Headache
  -> Dizziness
  -> Rash
  -> Dyspnoea
  -> Abdominal pain
  -> Constipation


In [9]:
demo_work = demo[["primaryid", "age", "age_cod", "sex"]].copy()

age_to_years = {
    "YR": 1,
    "DEC": 10,
    "MON": 1 / 12,
    "WK": 1 / 52,
    "DY": 1 / 365,
    "HR": 1 / (365 * 24),
}

demo_work["age_years"] = demo_work["age"] * demo_work["age_cod"].map(age_to_years)

invalid_age_mask = (demo_work["age_years"] <= 0) | (demo_work["age_years"] > 120)
invalid_age_count = invalid_age_mask.sum()
demo_work.loc[invalid_age_mask, "age_years"] = np.nan

demo_work["sex_clean"] = demo_work["sex"].replace({"UNK": "Unknown"}).fillna("Unknown")

demo_patient = demo_work[["primaryid", "age_years", "sex_clean"]].copy()

print("Patient feature table created.")
print(f"Rows: {len(demo_patient):,}")
print(f"Impossible ages set to missing: {invalid_age_count:,}")

print("\nPreview:")
display(demo_patient.head())

print("\nSex distribution:")
print(demo_patient["sex_clean"].value_counts())

print("\nAge summary (years):")
display(demo_patient["age_years"].describe())

Patient feature table created.
Rows: 400,514
Impossible ages set to missing: 254

Preview:


,primaryid,age_years,sex_clean
0,62150432,85.0,F
1,62645954,59.0,M
2,65308484,25.0,F
3,65388687,30.0,F
4,65418874,NaN,F



Sex distribution:
sex_clean
F          186231
M          131469
Unknown     82814
Name: count, dtype: int64

Age summary (years):


count    243290.000000
mean         54.411368
std          21.599063
min           0.000342
25%          40.000000
50%          59.000000
75%          71.000000
max         114.000000
Name: age_years, dtype: float64

In [10]:
drug_work = drug[["primaryid", "drug_seq", "drugname", "prod_ai", "role_cod", "route"]].copy()

MULTI_SEP = "\\"

def normalize_ingredient(text):
    """Return a cleaned, consistently formatted ingredient name."""
    if pd.isna(text):
        return np.nan
    text = " ".join(str(text).upper().split())
    parts = [part.strip() for part in text.split(MULTI_SEP) if part.strip()]
    return MULTI_SEP.join(sorted(parts))

raw_ingredient = drug_work["prod_ai"].fillna(drug_work["drugname"])
unique_raw = raw_ingredient.dropna().unique()
clean_lookup = {value: normalize_ingredient(value) for value in unique_raw}
drug_work["ingredient"] = raw_ingredient.map(clean_lookup)

drug_work["drugname"] = drug_work["drugname"].str.upper().str.strip()
drug_work["route"] = drug_work["route"].fillna("Unknown")

from_prod_ai = drug_work["prod_ai"].notna()

print(f"Distinct raw drug names              : {drug['drugname'].nunique():,}")
print(f"Distinct ingredients (from prod_ai)  : {drug_work.loc[from_prod_ai, 'ingredient'].nunique():,}")
print(f"Rows that used drugname as fallback  : {(~from_prod_ai).sum():,} "
      f"({(~from_prod_ai).mean() * 100:.1f}% of drug rows)")
print(f"Distinct fallback names              : {drug_work.loc[~from_prod_ai, 'ingredient'].nunique():,}")
print(f"Distinct values in total             : {drug_work['ingredient'].nunique():,}")

print("\nMost common ingredients:")
print(drug_work["ingredient"].value_counts().head(15))

Distinct raw drug names              : 29,057
Distinct ingredients (from prod_ai)  : 6,498
Rows that used drugname as fallback  : 31,933 (1.6% of drug rows)
Distinct fallback names              : 16,818
Distinct values in total             : 23,306

Most common ingredients:


ingredient
TIRZEPATIDE                 52320
DUPILUMAB                   47770
INFLIXIMAB-DYYB             35741
PREDNISONE                  28741
METHOTREXATE                23840
ACETAMINOPHEN               23654
TOCILIZUMAB                 21698
VEDOLIZUMAB                 21296
RITUXIMAB                   20019
ADALIMUMAB                  16512
OMALIZUMAB                  16430
INFLIXIMAB                  16107
CETIRIZINE HYDROCHLORIDE    15837
SULFASALAZINE               14074
FOLIC ACID                  13800
Name: count, dtype: int64


In [11]:
role_priority = {"PS": 1, "SS": 2, "C": 3, "I": 4}

drug_work = drug_work[drug_work["role_cod"] != "DN"].copy()
drug_work = drug_work.dropna(subset=["ingredient"]).copy()

drug_work["role_priority"] = drug_work["role_cod"].map(role_priority)
drug_work = drug_work.sort_values(
    ["primaryid", "role_priority", "drug_seq", "drugname", "route"],
    kind="stable",
)

drug_work = drug_work.drop_duplicates(subset=["primaryid", "ingredient"], keep="first").copy()

print("Drug cleaning complete.")
print(f"Rows after cleaning: {len(drug_work):,}")

print("\nRole distribution after cleaning:")
print(drug_work["role_cod"].value_counts())

print(f"\nReports with at least one drug: {drug_work['primaryid'].nunique():,}")

Drug cleaning complete.
Rows after cleaning: 1,302,326

Role distribution after cleaning:
role_cod
C     628175
PS    400514
SS    268774
I       4863
Name: count, dtype: int64

Reports with at least one drug: 400,514


In [12]:
drug_count_per_report = drug_work.groupby("primaryid").size()

print("Drug count per report:")
print(drug_count_per_report.describe())

print("\nReports by number of drugs:")
print(drug_count_per_report.value_counts().sort_index().head(15))

more_than_3 = (drug_count_per_report > 3).sum()
print(f"\nReports with more than 3 drugs: {more_than_3:,} "
      f"({more_than_3 / len(drug_count_per_report) * 100:.2f}% of reports)")

n_drugs = drug_count_per_report.rename("n_drugs").reset_index()

Drug count per report:
count    400514.000000
mean          3.251637
std           6.099711
min           1.000000
25%           1.000000
50%           1.000000
75%           3.000000
max         227.000000
dtype: float64

Reports by number of drugs:
1     245868
2      44235
3      25056
4      16103
5      11557
6       9034
7       7039
8       5906
9       5044
10      4200
11      3721
12      3062
13      2661
14      2169
15      1936
Name: count, dtype: int64

Reports with more than 3 drugs: 85,355 (21.31% of reports)


In [13]:
role_report_counts = drug_work.pivot_table(
    index="primaryid",
    columns="role_cod",
    values="ingredient",
    aggfunc="count",
    fill_value=0,
)

print("Reports containing each drug role")
for role in ["PS", "SS", "C", "I"]:
    if role in role_report_counts.columns:
        count = (role_report_counts[role] > 0).sum()
        percentage = count / len(role_report_counts) * 100
        print(f"{role}: {count:,} reports ({percentage:.2f}%)")

Reports containing each drug role
PS: 400,514 reports (100.00%)
SS: 71,391 reports (17.82%)
C: 108,829 reports (27.17%)
I: 1,928 reports (0.48%)


In [14]:
drug_work["drug_number"] = drug_work.groupby("primaryid").cumcount() + 1

three_drugs = drug_work[drug_work["drug_number"] <= 3].copy()

print("Top-3 drug selection complete.")
print(f"Rows retained: {len(three_drugs):,}")

print("\nDrug position counts:")
print(three_drugs["drug_number"].value_counts().sort_index())

print("\nExample reports:")
display(three_drugs[["primaryid", "drug_number", "ingredient", "drugname", "role_cod", "route"]].head(15))

Top-3 drug selection complete.


Rows retained: 665,571

Drug position counts:
drug_number
1    400514
2    154646
3    110411
Name: count, dtype: int64

Example reports:


,primaryid,drug_number,ingredient,drugname,role_cod,route
0,62150432,1,METHYLPREDNISOLONE SODIUM SUCCINATE,SOLU-MEDROL,PS,Intravenous (not otherwise specified)
1,62150432,2,PACLITAXEL,TAXOL,SS,Intravenous (not otherwise specified)
2,62150432,3,RANITIDINE HYDROCHLORIDE,RANITIDINE HYDROCHLORIDE,SS,Intravenous (not otherwise specified)
5,62645954,1,OXYCODONE HYDROCHLORIDE,OXYCONTIN,PS,Oral
6,62645954,2,SUNITINIB,SUNITINIB,SS,Oral
7,62645954,3,POLYETHYLENE GLYCOL 3350\POTASSIUM CHLORIDE\SO...,MOVICOLON,C,Unknown
9,65308484,1,SULFASALAZINE,SULFASALAZINE,PS,Unknown
10,65388687,1,ETANERCEPT,ENBREL,PS,Subcutaneous
13,65388687,2,HYDROXYCHLOROQUINE SULFATE,HYDROXYCHLOROQUINE SULFATE,C,Unknown
14,65388687,3,ADALIMUMAB,ADALIMUMAB,C,Unknown


In [15]:
drug_pivot = three_drugs.pivot(
    index="primaryid",
    columns="drug_number",
    values=["ingredient", "drugname", "role_cod", "route"],
)

drug_pivot.columns = [
    f"drug_{drug_number}_{feature}" for feature, drug_number in drug_pivot.columns
]

ordered_columns = [
    f"drug_{n}_{feature}" for n in (1, 2, 3) for feature in ("ingredient", "drugname", "role_cod", "route")
]
drug_pivot = drug_pivot[ordered_columns].reset_index()

print("Drug pivot table created.")
print(f"Rows: {len(drug_pivot):,}")
print(f"Columns: {drug_pivot.shape[1]}")

print("\nPreview:")
display(drug_pivot.head())

Drug pivot table created.
Rows: 400,514
Columns: 13

Preview:


,primaryid,drug_1_ingredient,drug_1_drugname,drug_1_role_cod,drug_1_route,drug_2_ingredient,drug_2_drugname,drug_2_role_cod,drug_2_route,drug_3_ingredient,drug_3_drugname,drug_3_role_cod,drug_3_route
0,62150432,METHYLPREDNISOLONE SODIUM SUCCINATE,SOLU-MEDROL,PS,Intravenous (not otherwise specified),PACLITAXEL,TAXOL,SS,Intravenous (not otherwise specified),RANITIDINE HYDROCHLORIDE,RANITIDINE HYDROCHLORIDE,SS,Intravenous (not otherwise specified)
1,62645954,OXYCODONE HYDROCHLORIDE,OXYCONTIN,PS,Oral,SUNITINIB,SUNITINIB,SS,Oral,POLYETHYLENE GLYCOL 3350\POTASSIUM CHLORIDE\SO...,MOVICOLON,C,Unknown
2,65308484,SULFASALAZINE,SULFASALAZINE,PS,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,65388687,ETANERCEPT,ENBREL,PS,Subcutaneous,HYDROXYCHLOROQUINE SULFATE,HYDROXYCHLOROQUINE SULFATE,C,Unknown,ADALIMUMAB,ADALIMUMAB,C,Unknown
4,65418874,ETANERCEPT,ENBREL,PS,Transplacental,METHOTREXATE,METHOTREXATE,C,Transplacental,HYDROXYCHLOROQUINE SULFATE,HYDROXYCHLOROQUINE SULFATE,C,Transplacental


In [16]:
reac_work = reac[["primaryid", "pt"]].copy()

for reaction in target_columns:
    reac_work[reaction] = (reac_work["pt"] == reaction).astype(int)

reac_targets = reac_work.groupby("primaryid")[target_columns].max().reset_index()

print("Reaction target table created.")
print(f"Rows: {len(reac_targets):,}")

print("\nTarget prevalence (number of reports, % of reports):")
prevalence = reac_targets[target_columns].sum().sort_values(ascending=False)
display(pd.DataFrame({
    "reports": prevalence,
    "percent": (prevalence / len(reac_targets) * 100).round(2),
}))

Reaction target table created.
Rows: 400,514

Target prevalence (number of reports, % of reports):


,reports,percent
Fatigue,18639,4.65
Nausea,15251,3.81
Diarrhoea,14891,3.72
Headache,12421,3.10
Dyspnoea,11692,2.92
Rash,10263,2.56
Vomiting,10089,2.52
Dizziness,9274,2.32
Constipation,5548,1.39
Abdominal pain,5146,1.28


In [17]:
ml_dataset = (
    demo_patient
    .merge(n_drugs, on="primaryid", how="left")
    .merge(drug_pivot, on="primaryid", how="left")
    .merge(reac_targets, on="primaryid", how="left")
)

ml_dataset["n_drugs"] = ml_dataset["n_drugs"].fillna(0).astype(int)
ml_dataset[target_columns] = ml_dataset[target_columns].fillna(0).astype(int)

print("ML dataset created.")
print(f"Rows: {len(ml_dataset):,}")
print(f"Columns: {ml_dataset.shape[1]:,}")

print("\nColumn names:")
print(ml_dataset.columns.tolist())

print("\nMissing values (top 15):")
display(ml_dataset.isna().sum().sort_values(ascending=False).head(15))

print("\nPreview:")
display(ml_dataset.head())

ML dataset created.
Rows: 400,514
Columns: 26

Column names:
['primaryid', 'age_years', 'sex_clean', 'n_drugs', 'drug_1_ingredient', 'drug_1_drugname', 'drug_1_role_cod', 'drug_1_route', 'drug_2_ingredient', 'drug_2_drugname', 'drug_2_role_cod', 'drug_2_route', 'drug_3_ingredient', 'drug_3_drugname', 'drug_3_role_cod', 'drug_3_route', 'Diarrhoea', 'Nausea', 'Vomiting', 'Fatigue', 'Headache', 'Dizziness', 'Rash', 'Dyspnoea', 'Abdominal pain', 'Constipation']

Missing values (top 15):


drug_3_ingredient    290103
drug_3_drugname      290103
drug_3_role_cod      290103
drug_3_route         290103
drug_2_route         245868
drug_2_role_cod      245868
drug_2_drugname      245868
drug_2_ingredient    245868
age_years            157224
primaryid                 0
drug_1_role_cod           0
drug_1_route              0
drug_1_ingredient         0
drug_1_drugname           0
n_drugs                   0
dtype: int64


Preview:


,primaryid,age_years,sex_clean,n_drugs,drug_1_ingredient,drug_1_drugname,drug_1_role_cod,drug_1_route,drug_2_ingredient,drug_2_drugname,drug_2_role_cod,drug_2_route,drug_3_ingredient,drug_3_drugname,drug_3_role_cod,drug_3_route,Diarrhoea,Nausea,Vomiting,Fatigue,Headache,Dizziness,Rash,Dyspnoea,Abdominal pain,Constipation
0,62150432,85.0,F,5,METHYLPREDNISOLONE SODIUM SUCCINATE,SOLU-MEDROL,PS,Intravenous (not otherwise specified),PACLITAXEL,TAXOL,SS,Intravenous (not otherwise specified),RANITIDINE HYDROCHLORIDE,RANITIDINE HYDROCHLORIDE,SS,Intravenous (not otherwise specified),0,0,0,0,0,0,0,0,0,0
1,62645954,59.0,M,4,OXYCODONE HYDROCHLORIDE,OXYCONTIN,PS,Oral,SUNITINIB,SUNITINIB,SS,Oral,POLYETHYLENE GLYCOL 3350\POTASSIUM CHLORIDE\SO...,MOVICOLON,C,Unknown,0,0,0,0,0,0,0,0,0,0
2,65308484,25.0,F,1,SULFASALAZINE,SULFASALAZINE,PS,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0
3,65388687,30.0,F,17,ETANERCEPT,ENBREL,PS,Subcutaneous,HYDROXYCHLOROQUINE SULFATE,HYDROXYCHLOROQUINE SULFATE,C,Unknown,ADALIMUMAB,ADALIMUMAB,C,Unknown,0,0,0,0,0,0,0,0,0,0
4,65418874,NaN,F,12,ETANERCEPT,ENBREL,PS,Transplacental,METHOTREXATE,METHOTREXATE,C,Transplacental,HYDROXYCHLOROQUINE SULFATE,HYDROXYCHLOROQUINE SULFATE,C,Transplacental,0,0,0,0,0,0,0,0,0,0


In [18]:
assert ml_dataset["primaryid"].is_unique, "Each report should appear only once"
assert ml_dataset["age_years"].dropna().between(0, 120).all(), "Ages must be between 0 and 120"
assert ml_dataset[target_columns].isin([0, 1]).all().all(), "Targets must be 0 or 1"
assert ml_dataset["drug_1_ingredient"].notna().mean() > 0.99, "Almost every report should have a first drug"

print("All sanity checks passed.")

All sanity checks passed.


In [19]:
processed_path = PROCESSED_DATA_DIR / "faers_ml_dataset.csv"

ml_dataset.to_csv(processed_path, index=False)

file_size_mb = processed_path.stat().st_size / (1024 ** 2)

print("Processed dataset saved.")
print(f"Path : {processed_path}")
print(f"Size : {file_size_mb:.2f} MB")
print(f"Shape: {ml_dataset.shape}")

Processed dataset saved.
Path : C:\Users\Krish\Desktop\BU\Codes\Projects\ADR\adr-prediction-system\data\processed\faers_ml_dataset.csv
Size : 43.08 MB
Shape: (400514, 26)
